# 02 Exploratory Data Analysis

Project: Predicting Thyroid Dysfunction Using Machine Learning

Author: Namitha Nair

This notebook is part of a cleaned research workflow for the NHANES thyroid-metabolic analysis.


## Purpose

This notebook performs exploratory data analysis (EDA) on the merged NHANES thyroid-metabolic dataset. It includes missingness checks, summary statistics, TSH distribution, correlations, and group-level comparisons.


In [ ]:
from pathlib import Path

# Project paths. These work if the notebooks are run from the repository root.
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
FIGURES_DIR = Path("figures")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = PROCESSED_DIR / "NHANES_Thyroid_Glucose_Dataset.csv"
df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()


## Create analysis dataset

For the core analysis, participants must have non-missing thyroid and metabolic measurements.


In [ ]:
analysis_df = df.dropna(
    subset=[
        "TSH",
        "Free_T3",
        "Free_T4",
        "Fasting_Glucose_mg_dL",
        "HbA1c_Percent",
        "BMI"
    ]
).copy()

print("Analysis dataset shape:", analysis_df.shape)


In [ ]:
analysis_df.describe()


## Correlation matrix


In [ ]:
corr_vars = [
    "BMI",
    "Waist_Circumference_cm",
    "Fasting_Glucose_mg_dL",
    "Fasting_Insulin_uU_mL",
    "HbA1c_Percent",
    "TSH",
    "Free_T3",
    "Free_T4"
]

corr_matrix = analysis_df[corr_vars].corr()
corr_matrix


In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(corr_matrix, aspect="auto")
plt.colorbar(label="Pearson correlation")
plt.xticks(range(len(corr_vars)), corr_vars, rotation=90)
plt.yticks(range(len(corr_vars)), corr_vars)
plt.title("Correlation Matrix of Thyroid and Metabolic Variables")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "correlation_matrix.png", dpi=300)
plt.show()


## TSH distribution


In [ ]:
plt.figure(figsize=(8, 5))
analysis_df["TSH"].hist(bins=100)
plt.title("Distribution of TSH")
plt.xlabel("TSH")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "tsh_distribution.png", dpi=300)
plt.show()

analysis_df["TSH"].describe(percentiles=[0.9, 0.95, 0.99])


## Define TSH groups

Clinical threshold grouping used for exploratory analysis:

- Low TSH: `< 0.4`
- Normal TSH: `0.4–4.0`
- High TSH: `> 4.0`


In [ ]:
def tsh_group(tsh):
    if tsh < 0.4:
        return "Low"
    elif tsh > 4.0:
        return "High"
    else:
        return "Normal"

analysis_df["TSH_Group"] = analysis_df["TSH"].apply(tsh_group)

counts = analysis_df["TSH_Group"].value_counts()
percentages = analysis_df["TSH_Group"].value_counts(normalize=True) * 100

pd.DataFrame({"Count": counts, "Percent": percentages.round(2)})


## Group summaries


In [ ]:
summary_vars = [
    "Fasting_Glucose_mg_dL",
    "HbA1c_Percent",
    "BMI",
    "Fasting_Insulin_uU_mL"
]

mean_table = analysis_df.groupby("TSH_Group")[summary_vars].mean()
median_table = analysis_df.groupby("TSH_Group")[summary_vars].median()

print("Mean table")
display(mean_table)

print("Median table")
display(median_table)


In [ ]:
for var in summary_vars:
    analysis_df.boxplot(column=var, by="TSH_Group")
    plt.title(f"{var} by TSH Group")
    plt.suptitle("")
    plt.xlabel("TSH Group")
    plt.ylabel(var)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"boxplot_{var}_by_tsh_group.png", dpi=300)
    plt.show()


In [ ]:
# Save analysis dataset with TSH groups for downstream notebooks.
output_path = PROCESSED_DIR / "NHANES_Analysis_Dataset.csv"
analysis_df.to_csv(output_path, index=False)
print(f"Saved analysis dataset to: {output_path}")
